# BetHero — Cards_3.5 full-budget training (single market, full dataset)

One account, one GPU, one market — trains **Cards_3.5** on the full feature-cache dataset (all eligible rows, no row cap) at `n_trials=20`.

This is one of 8 identical notebooks (one per market_key) meant to run in parallel across separate Google accounts, each with its own free Colab GPU — see the other `train_<market>.ipynb` files in `notebooks/`.

**One-time setup before running this notebook:**
1. This Google account needs its own copy of `data_snapshot/` (131MB) uploaded to `MyDrive/bethero/data_snapshot/`.
2. Add three Colab secrets (key icon in the left sidebar): `SUPABASE_URL`, `SUPABASE_SERVICE_ROLE_KEY`, and `GITHUB_TOKEN` (fine-grained token scoped to this repo, "Contents: Read-only" — needed because the repo is private).
3. Runtime → Change runtime type → **T4 GPU**.

The trained model lands directly in this account's Drive at `MyDrive/bethero/models/` — once finished, download/zip it and merge it into the shared local `models/` directory alongside the other 7 markets' results from their own accounts.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Repo is private, so cloning needs a GitHub token — add one as a Colab
# secret named GITHUB_TOKEN (key icon, left sidebar), not pasted here.
# Create it at https://github.com/settings/tokens (fine-grained token,
# scoped to just this repo, "Contents: Read-only" permission is enough).
from google.colab import userdata

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/joeie5/bethero.git"

!git clone $REPO_URL /content/bethero
%cd /content/bethero

In [ ]:
# Only what training actually needs — not the full app (FastAPI, Celery,
# Anthropic/Gemini clients, NewsAPI, etc. are irrelevant here). Mostly
# unpinned: requirements.txt's exact pins don't have prebuilt wheels for
# Colab's Python version, so pip was falling back to slow from-source
# builds (20+ minutes and still going) — confirmed live. Also skip torch
# entirely — Colab already ships a CUDA-enabled build.
# supabase IS pinned though — app/database.py builds a ClientOptions(...)
# with fields (storage_client_timeout) that a newer supabase-py version
# restructured, confirmed live: "'ClientOptions' object has no attribute
# 'storage'". supabase-py itself is pure Python, so pinning it doesn't
# reintroduce the slow-build problem the other packages had.
# sqlalchemy is required by app/database.py but isn't even listed in
# requirements.txt at all — a pre-existing gap, added here explicitly.
!pip install -q pandas numpy scikit-learn xgboost lightgbm joblib optuna mlflow "supabase==2.10.0" python-dotenv sqlalchemy pydantic pydantic-settings

In [ ]:
import os
from google.colab import userdata

os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_SERVICE_ROLE_KEY"] = userdata.get("SUPABASE_SERVICE_ROLE_KEY")

# This account's own Drive — no collision risk with the other 7 markets'
# accounts since each has a completely separate Drive.
DRIVE_MODEL_DIR = "/content/drive/MyDrive/bethero/models"
os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
os.environ["MODEL_DIR"] = DRIVE_MODEL_DIR

DATA_DIR = "/content/drive/MyDrive/bethero/data_snapshot"
assert os.path.isdir(DATA_DIR), f"Upload data_snapshot/ to {DATA_DIR} first"
print("MODEL_DIR ->", DRIVE_MODEL_DIR)
print("DATA_DIR  ->", DATA_DIR)

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only — check Runtime > Change runtime type")

In [ ]:
import sys
sys.path.insert(0, "/content/bethero")

from pathlib import Path
from app.ml.models import MODEL_DIR
from app.ml.training_pipeline_logic import train_market

MODEL_KEYS = ["Cards_3.5"]   # this notebook trains ONLY this market
N_TRIALS = 20
N_META_TRIALS = 20

# Full dataset — no row cap. This account is dedicated to one market only,
# so there's no time-sharing tradeoff like the original all-8-in-one-loop
# notebook had. Real per-market estimate at n_trials=20, full rows:
# ~6.2 hours for full-row markets (BTTS/Goals*/1x2), ~2.7-2.8 hours for
# Cards_3.5/Corners* (fewer eligible rows — some Extra-tier countries lack
# corners/cards data, per CLAUDE.md §5, dropped not zero-filled).
LIMIT = None

print("MODEL_DIR resolved to:", MODEL_DIR)
assert str(MODEL_DIR) == os.environ["MODEL_DIR"], "MODEL_DIR env var must be set BEFORE importing app.ml.* — restart runtime and re-run from the top"
print(f"Training market: Cards_3.5  |  limit: {LIMIT}")

In [ ]:
import time

for model_key in MODEL_KEYS:
    ensemble_path = MODEL_DIR / f"StackingEnsemble_football_{model_key}.joblib"
    if ensemble_path.exists():
        print(f"=== {model_key}: already trained, skipping ({ensemble_path}) ===")
        continue

    print(f"\n=== {model_key} — starting (limit={LIMIT}) ===", flush=True)
    t0 = time.monotonic()
    results = train_market(
        "football", model_key,
        n_trials=N_TRIALS, n_meta_trials=N_META_TRIALS,
        limit=LIMIT,
        data_dir=Path(DATA_DIR),
    )
    elapsed = time.monotonic() - t0
    for r in results:
        print(f"  {r.model_name}: {r.status} {r.error or ''} ({r.duration_s}s)")
        if r.calibration_table is not None:
            print(r.calibration_table.to_string(index=False))
    print(f"{model_key} done in {elapsed/60:.1f} min", flush=True)

print("\n=== ALL MARKETS DONE ===")

## After this finishes
`Cards_3.5`'s trained model files are sitting in this account's Drive at `MyDrive/bethero/models/`. Download/zip that folder and merge it into the shared local `models/` directory (same process used for Phase A/B) — it'll sit alongside the other 7 markets' results once each account finishes its own.